In [41]:
%pip install neo4j
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [42]:
import json
import os
import requests
import time
from tqdm import tqdm
from neo4j import GraphDatabase

In [43]:
class Neo4jConnection:

    def __init__(self, uri, user, pwd):
        self.__uri = uri
        self.__user = user
        self.__pwd = pwd
        self.__driver = None
        try:
            self.__driver = GraphDatabase.driver(self.__uri, auth=(self.__user, self.__pwd))
        except Exception as e:
            print("Failed to create the driver:", e)

    def close(self):
        if self.__driver is not None:
            self.__driver.close()

    def query(self, query, parameters=None, db=None):
        assert self.__driver is not None, "Driver not initialized!"
        session = None
        response = None
        try:
            session = self.__driver.session(database=db) if db is not None else self.__driver.session()
            response = list(session.run(query, parameters))
        except Exception as e:
            print("Query failed:", e)
        finally:
            if session is not None:
                session.close()
        return response

In [44]:
def readlines(infile):
  with open(infile) as file:
    return file.readlines()
  


def getPath(fileName):
  return os.path.join(os.path.dirname(__file__), fileName)


def accessAlbum(albumName, albumArtist):
  url = "https://musicbrainz.org/ws/2/release"
  params = {
    "query": f'release:"{albumName}" AND artist:"{albumArtist}"',
    "fmt": "json",
    "limit": 1
  }

  headers = {"User-Agent": "1001_Albums_Recommender/1.0 (roxyknit2000@gmail.com)"}
  response = requests.get(url, params=params, headers=headers)
  data = response.json()
  print(data)

  if not data["releases"]:
        print(f"No results found for {albumName} by {albumArtist}")
        return None

  return data["releases"][0]["id"]
  

def accessSongs(albumId):
  url = f"https://musicbrainz.org/ws/2/release/{albumId}"
  params = {
    "inc": "recordings",
    "fmt": "json"
  }

  headers = {"User-Agent": "1001_Albums_Recommender/1.0 (roxyknit2000@gmail.com)"}

  response = requests.get(url, params=params, headers=headers)
  data = response.json()
  
  tracks = []
  for track in data["media"][0]["tracks"]:
      tracks.append({
        "title": track["title"],
        "duration": track.get("length", -1),
        "mbid": track["recording"]["id"]
      })
      
  return tracks


def addSongsToDB(albumName, albumYear, songs, connection):
  for song in songs:

    addSongQuery = """
MATCH (al:Album {name: $albumName, year: $albumYear})
MERGE (s:Song {name: $title, mbid: $mbid})
SET s.song_length = $duration
MERGE (al)-[:Lists]->(s)
"""

    connection.query(addSongQuery, parameters={
      "albumName": albumName, 
      "albumYear": albumYear, 
      "title":song["title"],
      "duration": song["duration"], 
      "mbid": song["mbid"]})




def replaceNodeNames(oldAlbumName, oldArtist, albumName, albumArtist, conn):
    print(f"Trying to update album: '{oldAlbumName}' -> '{albumName}'")
    print(f"Trying to update artist: '{oldArtist}' -> '{albumArtist}'")

    updateAlbumNameQuery = """
MATCH (al:Album {name: $oldAlbumName})
SET al.name = $albumName
RETURN al
"""
    result = conn.query(updateAlbumNameQuery, parameters={
       "oldAlbumName": oldAlbumName,
       "albumName":albumName,
    })
    print(f"Album update result: {result}")

    updateArtistNameQuery = """
MATCH (ar:Artist{name:$oldArtist})
SET ar.name = $albumArtist
RETURN ar
"""

    result = conn.query(updateArtistNameQuery, parameters={
       "oldArtist": oldArtist,
       "albumArtist": albumArtist
    })
    print(f"Artist update result: {result}")



def getAlbumInfo(albumId):
    url = f"https://musicbrainz.org/ws/2/release/{albumId}"
    params = {
        "inc": "artist-credits",
        "fmt": "json"
    }
    headers = {"User-Agent": "1001_Albums_Recommender/1.0 (roxyknit2000@gmail.com)"}
    
    response = requests.get(url, params=params, headers=headers)
    data = response.json()
    print(data)
    
    albumName = data["title"]
    artist = data["artist-credit"][0]["artist"]["name"]
    
    return albumName, artist


def deleteAlbumNode(albumName, conn):
   deleteAlbumQuery = """
MATCH (al:Album {name: $albumName})
DETACH DELETE al
"""
   conn.query(deleteAlbumQuery, parameters={"albumName": albumName})
   print(f"{albumName} deleted from db")


def openConnection():
   with open("config.json") as configFile:
    config = json.load(configFile)
  
    conn= Neo4jConnection(uri= config["uriNeo4j"], user=config["userNeo4j"], pwd=config["passwordNeo4j"])
    return conn


In [45]:
# -----------------------------------------------------------------------------
# Adding albums, artists, and songs to Neo4j DB
# -----------------------------------------------------------------------------

conn = openConnection()


# with open("1001-albums-you-must-hear-before-you-die.csv") as file:
#   albumLines = file.readlines()


# for line in albumLines[1:]:
#   line_info = line.strip().split("|")
#   addRelationshipQuery = """
# MERGE (al:Album {name: $albumName, year: $year})
# MERGE(ar:Artist {name: $artistName})
# MERGE (ar)-[ :RELEASED]->(al)
# """

#   conn.query(addRelationshipQuery, parameters={
#     "albumName": line_info[1], "year": int(line_info[0]), "artistName": line_info[2]
#     })
  

# Initial pass through lines in album
# with open("unsearchable.txt", "w") as file:
#   for line in albumLines[1:]:
#     line_info = line.strip().split("|")
#     albumName, albumArtist, albumYear = line_info[1], line_info[2], line_info[0]
#     print(albumName)
#     albumId = accessAlbum(albumName, albumArtist)
#     print(albumId)
#     if not albumId:
#       file.write(f"{albumArtist}|{albumName}|{albumYear}\n")
#       continue
#     time.sleep(1)
#     songs = accessSongs(albumId)
#     print(songs)
#     addSongsToDB(albumName, int(albumYear), songs, conn)


# # Update nodes with underscores and add songs for those albums
# with open("unsearchable.txt") as file:
#   notFoundLines = file.readlines()

# with open("still_unsearchable.txt", "w") as file:
#   for line in notFoundLines:
#     if not "_" in line:
#       file.write(line)
#       continue
    
#     oldArtist, oldName, oldYear = line.strip().split("|")
#     albumArtist, albumName, albumYear = line.replace("_", "'").strip().split("|")
#     albumId = accessAlbum(albumName, albumArtist)
#     print(albumId)
#     if not albumId:
#       file.write(line)
#       continue
#     replaceNodeNames(oldName, oldArtist, albumName, albumArtist, conn)
#     time.sleep(1)
#     songs = accessSongs(albumId)
#     print(songs)
#     addSongsToDB(albumName, int(albumYear), songs, conn)


# with open("still_unsearchable.txt") as unsearchable, open("narrowedAlbumIds.txt") as idFile:
#   unsearchableLines = unsearchable.readlines()
#   albumIds = idFile.readlines()

# # # Add remaining albums and update the node names with the albumIDs directly
# with open("leftoverAlbums.txt", "w") as file:
#   for oldAlbumInfo, albumId in zip(unsearchableLines, albumIds):
#     if not oldAlbumInfo.strip():
#       break
#     oldArtist, oldName, oldYear = oldAlbumInfo.strip().split("|")
#     print(oldName)
#     if "NOT AVAILABLE" in albumId:
#       deleteAlbumNode(oldName, conn)
#       file.write(oldAlbumInfo)
#       continue
    
#     albumId = albumId.strip()
#     newAlbum, newArtist = getAlbumInfo(albumId)
#     replaceNodeNames(oldName, oldArtist, newAlbum, newArtist, conn)
#     time.sleep(1)
#     songs = accessSongs(albumId)
#     print(songs)
#     addSongsToDB(newAlbum, int(oldYear), songs, conn)


print("Done!")
conn.close()

Done!


In [46]:
# ---------------------------------------------------------------------
# Helper functions for retrieving songs from last few albums
# ---------------------------------------------------------------------

def getEmptyAlbums(conn):
  getAlbumsWithoutSongsQuery = """
MATCH (ar:Artist)-[:RELEASED]->(al:Album)
WHERE NOT (al)-[:Lists]->(:Song)
RETURN al.name, al.year, ar.name
"""
  result = conn.query(getAlbumsWithoutSongsQuery)
  for album in result:
    print(album["al.name"], album["al.year"])
  return [(album["al.name"], album["al.year"], album["ar.name"]) for album in result]


In [47]:
# ---------------------------------------------------------------------
# Add songs for any albums that have not yet added songs.
# ---------------------------------------------------------------------

conn = openConnection()

emptyAlbums = getEmptyAlbums(conn)

for album, year, artist in emptyAlbums:
  print(f"{album}, {artist}")

  albumId = accessAlbum(album, artist)
  time.sleep(1)
  songs = accessSongs(albumId)
  print(songs)
  addSongsToDB(album, int(year), songs, conn)



print("Done!")
conn.close()

Done!


In [ ]:
# ----------------------------------------------------------------------
# Helper Functions for adding song genres
# ----------------------------------------------------------------------

def getSongsFromDB(conn):
  songsQuery = """
MATCH (s:Song)
RETURN s.name, s.mbid
"""

  result = conn.query(songsQuery)
  songInfo = [(record["s.name"], record["s.mbid"]) for record in result]
  return songInfo


def get_song_genres(song_mbid):
    url = f"https://musicbrainz.org/ws/2/recording/{song_mbid}"
    params = {
        "inc": "genres",
        "fmt": "json"
    }
    headers = {"User-Agent": "MyWebsite/1.0 (myemail@gmail.com)"}
    
    response = requests.get(url, params=params, headers=headers)
    data = response.json()
    genres = [genre["name"] for genre in data["recording"].get("genres", [])]
    print(genres)
    return genres

def addGenresToSongNode():
   pass


In [ ]:
conn = openConnection()

songInfo = getSongsFromDB(conn)

for songName, songId in songInfo:
  print(songName)
  

conn.close()

11446
('In the Wee Small Hours of the Morning', '69ebc4ab-2eb3-4bbd-9c2f-4437d40d4868')
('Mood Indigo', '6b50d0c4-1f6b-459f-a05a-bcba82666c85')
('Glad to Be Unhappy', '4b778a78-0d52-41b9-bcd8-efab41661f80')
('I Get Along Without You Very Well', '80a7fb9c-edf8-49aa-a900-6cdcca3529db')
('Deep in a Dream', 'deae8ec0-cba2-4d71-9738-2739df6f021e')
('I See Your Face Before Me', '53515490-9dda-4cb9-9c5c-f9f1a6cde71d')
('Can’t We Be Friends?', 'c68e7550-a76e-4fa5-8bf1-f153e8de4b20')
('When Your Lover Has Gone', 'a25e7f9a-8084-49f8-9755-e92031bb6055')
('What Is This Thing Called Love', '27422a14-0c51-4cae-9e86-2fc3fff0739a')
('Last Night When We Were Young', '244ae826-7803-4eca-805f-d46132b9dc61')
('I’ll Be Around', 'dc9b856b-4850-45db-b452-a19585a9a9c5')
('Ill Wind', '6047e7c6-3f56-47e1-b1eb-3128b1cadcfb')
('It Never Entered My Mind', 'fa862772-97b6-4071-a29d-0a0f0ed96077')
('Dancing on the Ceiling', '988e67e5-2f68-4ba9-90ed-e9c966728b14')
('I’ll Never Be the Same', 'bd79549d-29a4-410f-8f52-c7